# **Trabalho (Parte 1)**

# Elementos do Grupo:
**Grupo 13:**
* Gonçalo Filipe Carvalho Conceição - 2025115285
* Joana Francisca Brandão Tomé - a2025109971
* Pedro Miguel Gaspar Cebolo - 2025116836

### Justificação da Escolha do Dataset
Para este trabalho de tratamento e análise de dados, selecionámos um dataset que simula casos de corrupção e fraude económica. A escolha foi estratégica: pedimos intencionalmente a uma plataforma de inteligência artificial para gerar um conjunto de dados com o maior número de erros, inconsistências e falhas de formatação possíveis.

O dataset escolhida refere-se à corrupção em portugal, baseado em notícias reais.

### Interesse no Tópico
O tema subjacente – Corrupção, Fraude e Accountability – é de extrema relevância social e económica. Embora os dados sejam sintéticos, eles refletem a complexidade de bases de dados reais que monitorizam irregularidades financeiras.

O nosso interesse reside em:

**Validar a Integridade**: Garantir que métricas críticas como montantes de suborno e número de acusados são registadas corretamente (algo que corrigimos ao imputar '0' em vez de usar a média, por exemplo).

**Padronizar Variáveis Categóricas**: Criar um vocabulário uniforme para regiões, setores e estado de investigação.

**Preparar para Análise Temporal**: Corrigir as datas (muitas não correspondiam).

### Problemas Estatísticos que Podem Ser Resolvidos
Após a limpeza intensiva executada no código Python, este dataset oferece oportunidades para explorar e resolver diversos problemas estatísticos no domínio da corrupção:

**Análise Exploratória e Tendências (EDA)**: Identificar os setores mais reportados (sector), as regiões com maior volume de casos (region) e as profissões mais frequentemente acusadas (accused_professions_list).

**Estudos de Correlação e Impacto**: Verificar se existe associação entre o montante total da fraude (amount_eur) e a duração da sentença (sentence_years), ou entre a exitência de suborno (bribe_flag) e de condenação (convicted_flag).

**Modelagem Preditiva (Classificação)**: Construir um modelo para prever a probabilidade de um caso ser arquivado vs. investigado (investigation_status) com base nas características iniciais do caso.

**Análise Temporal**: Estudar a distribuição temporal dos casos para identificar picos de reportes, possivelmente correlacionados com anos eleitorais ou alterações legislativas.

# 1. Preâmbulo

In [99]:
import pandas as pd
import numpy as np
pd.set_option('future.no_silent_downcasting', True)

## 2. Importar dados

In [56]:
df = pd.read_csv('corruption_dataset.csv')
df.shape

(50320, 21)

# 3. Primeira inspeção dos dados

> **Inspeção das primeiras e das últimas linhas.**

In [57]:
df.head()

,case_id,year_reported,region,municipality,sector,amount_eur,bribe_flag,bribe_amount,investigation_status,accused_count,...,convicted_flag,sentence_years,appeal_flag,case_description,source_url,reporter,media_outlet,date_reported,file_number,investigator_id
0,CT-004970,2022,"Porto,",Faro,Política,4197659.22,NaN,9492.44,Arquivado,2,...,False,NaN,Y,Operação policial com buscas e apreensões. Há ...,www.jornal-example.pt/article/3303,Miguel,Expresso,1997-02-03,Case 104969,INV029
1,CT-036506,2009,Braga,Funchal,Judiciário,1754118.0,False,NaN,Em investigação,2,...,False,NaN,Y,"Publicidade de contratos públicos, possível co...",htp:/ /invalid-url 4808,João,RTP,2003-12-03,Case 136505,INV225
2,CT-044321,1998,Porto,Setúbal,Energia,429175.02,False,NaN,Closed,3,...,False,NaN,Não,"Publicidade de contratos públicos, possível co...",htp:/ /invalid-url 9108,Miguel,RTP,29/09/2018,Case 144320,INV093
3,CT-049413,2007,Braga,Ponta Delgada,Transportes,50459.72,False,NaN,Em investigação,1,...,False,NaN,Não,Caso relativo a fraudes em licitações. Ver det...,www.jornal-example.pt/article/4315,Luís,TSF,16-Dec-2000,Case 149412,INV078
4,CT-014459,2011,Faro,Braga,Saúde,??4725049.85,False,NaN,Aberto,2,...,No,NaN,Não,Acusados negam as acusações. Processo com falt...,http://www.jornal-example.pt/article?id=15659,João,Público,18-Jul-2004,Case 114458,NaN


In [58]:
df.tail()

,case_id,year_reported,region,municipality,sector,amount_eur,bribe_flag,bribe_amount,investigation_status,accused_count,...,convicted_flag,sentence_years,appeal_flag,case_description,source_url,reporter,media_outlet,date_reported,file_number,investigator_id
50315,CT-011285,1996,Açores,Ponta Delgada,Privado,1740204.53,True,137.141.22,Fechado,3,...,Yes,8.5,Não,Denúncia sobre adjudicação directa a empresa X...,www.jornal-example.pt/article/14460,Miguel,RTP,01-Dec-2020,Case 111284,INV080
50316,CT-044733,2008,Braga,Setúbal,Privado,3949304.3,False,NaN,Closed,3,...,False,NaN,Não,"Publicidade de contratos públicos, possível co...",htp:/ /invalid-url 7707,Beatriz,RTP,23-Mar-1999,Case 144732,NaN
50317,CT-038159,2016,Coimbra,Porto,Judiciário,3777984.92,True,276.684.41,Closed,NaN,...,Nao,NaN,Sim,Acusados negam as acusações. Processo com falt...,htp:/ /invalid-url 375,Ana,Público,05-Mar-2001,Case 138158,INV020
50318,CT-000861,2022,Setúbal,Setúbal,Judiciário,78311.1,False,1004.46,Encerrado,3,...,False,NaN,Sim,Operação policial com buscas e apreensões. Há ...,https://news.example.com/pt/1729,Beatriz,Observador,26-Apr-2006,Case 100860,INV090
50319,CT-015796,2021,Braga,Setúbal,Judiciário,-3574800.06,False,NaN,Closed,3,...,False,NaN,N,Denúncia sobre adjudicação directa a empresa X...,htp:/ /invalid-url 3065,Beatriz,RTP,16-Mar-2017,Case 115795,INV261


> **Inspeção dos formatos e valores em falta.**

In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50320 entries, 0 to 50319
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   case_id               50300 non-null  object
 1   year_reported         50320 non-null  object
 2   region                50320 non-null  object
 3   municipality          50300 non-null  object
 4   sector                50320 non-null  object
 5   amount_eur            49310 non-null  object
 6   bribe_flag            48771 non-null  object
 7   bribe_amount          15693 non-null  object
 8   investigation_status  49349 non-null  object
 9   accused_count         49308 non-null  object
 10  accused_professions   50320 non-null  object
 11  convicted_flag        47871 non-null  object
 12  sentence_years        5506 non-null   object
 13  appeal_flag           47777 non-null  object
 14  case_description      50320 non-null  object
 15  source_url            47728 non-null

**Comentários:**

*  **Existem vários atributos com valores em falta.**

* Todas as categprias estão consideradas como object, sendo que algumas têm de ser alteradas. Isto irá ser feito juntamente com a limpeza

* A correção do formato das variáveis categóricas **só deve ser feita se for vantajosa para a análise**.

* As variáveis numéricas **não** estão no formato correto.

* **'year_reported' não está na mesma linha temporal de 'date_reported'**. Vamos proceder à uniformização.

# 4. Correção de formatos e início da limpeza

In [60]:
df['case_description'] = df['case_description'].str.replace(r'<script.*?>.*?</script>', '', regex=True)
df.columns = df.columns.str.lower().str.strip()

In [61]:
df['year_reported'] = pd.to_numeric(df['year_reported'], errors='coerce').fillna(0).astype(int)
df = df[df['year_reported'] > 0]

In [62]:
for col in ['region', 'municipality']:
    # Remover vírgulas e espaços iniciais/finais
    df[col] = df[col].astype(str).str.replace(',', '', regex=False).str.strip()
    # Corrigir acentuações e erros comuns (ex: 'Pôrto' -> 'Porto')
    df[col] = df[col].str.replace('Pôrto', 'Porto', regex=False)
    # Padronizar para maiúsculas para facilitar a correspondência
    df[col] = df[col].str.upper()

In [63]:
df['sector'] = df['sector'].astype(str).str.strip().str.upper()
# Substituir valores comuns em falta
df['sector'].replace('NAN', 'DESCONHECIDO')
df['sector'].replace('N/A', 'DESCONHECIDO')

0           POLÍTICA
1         JUDICIÁRIO
2            ENERGIA
3        TRANSPORTES
4              SAÚDE
            ...     
50315        PRIVADO
50316        PRIVADO
50317     JUDICIÁRIO
50318     JUDICIÁRIO
50319     JUDICIÁRIO
Name: sector, Length: 50300, dtype: object

In [64]:
def clean_professions(text):
    if pd.isna(text) or text.strip().upper() in ['NAN', 'N/A', '']:
        return []
    # Remover parênteses, apóstrofos e aspas, e dividir por vírgula
    text = text.replace('(', '').replace(')', '').replace("'", '').replace('"', '').strip()
    # Se já vier formatado como lista (p. ex., 'Advogado, Político')
    professions = [p.strip().upper() for p in text.split(',') if p.strip()]
    return professions

df['accused_professions_list'] = df['accused_professions'].apply(clean_professions)
df.drop('accused_professions', axis=1, inplace=True)

In [65]:
df['source_url'] = df['source_url'].astype(str).str.strip()
# Substituir 'htp:/ /' por 'http://'
df['source_url'] = df['source_url'].str.replace('htp:/ /', 'http://', regex=False)
# Remover qualquer tag HTML/script que possa ter sido deixada
df['source_url'] = df['source_url'].str.replace(r'<.*?>', '', regex=True)
# URLs inválidas ou em falta são preenchidas com NaN
df['source_url'].replace(['nan', 'n/a', ''], np.nan)

0                   www.jornal-example.pt/article/3303
1                              http://invalid-url 4808
2                              http://invalid-url 9108
3                   www.jornal-example.pt/article/4315
4        http://www.jornal-example.pt/article?id=15659
                             ...                      
50315              www.jornal-example.pt/article/14460
50316                          http://invalid-url 7707
50317                           http://invalid-url 375
50318                 https://news.example.com/pt/1729
50319                          http://invalid-url 3065
Name: source_url, Length: 50300, dtype: object

In [66]:
df['bribe_flag'] = df['bribe_flag'].astype(str).str.lower().str.strip()
df['convicted_flag'] = df['convicted_flag'].astype(str).str.lower().str.strip()
def to_boolean(series):
    return series.map({
        'true': True, 't': True, 'yes': True, 'y': True, 'sim': True, 's': True,
        'false': False, 'f': False, 'no': False, 'n': False, 'nao': False, 'não': False,
        '': False, 'nan': False, 'n/a': False
    }).fillna(False) # Valores desconhecidos ou em falta são assumidos como False

df['bribe_flag'] = to_boolean(df['bribe_flag'])
df['convicted_flag'] = to_boolean(df['convicted_flag'])

In [80]:
df['bribe_amount'] = df['bribe_amount'].astype(str).str.replace(',', '.', regex=False)
df['bribe_amount'] = pd.to_numeric(df['bribe_amount'], errors='coerce')

# aqui sim substitui os NaN por 0
df['bribe_amount'] = df['bribe_amount'].fillna(0)

df['bribe_amount'] = df['bribe_amount'].round(2)

In [68]:
df['accused_count'] = pd.to_numeric(df['accused_count'], errors='coerce')
df['accused_count'] = df['accused_count'].fillna(0).astype(int)

In [69]:
df['sentence_years'] = pd.to_numeric(df['sentence_years'], errors='coerce').fillna(0).round(1)

In [70]:
df['appeal_flag'] = to_boolean(df['appeal_flag'].astype(str).str.strip().str.lower())

In [71]:
def parse_date_simple(x):
    x = str(x).strip()
    if x.lower() in ['nan', '', 'none', 'n/a', 'null']:
        return pd.NaT
    formats = [
        '%Y-%m-%d', '%d/%m/%Y', '%d-%m-%Y', '%d-%b-%Y',
        '%Y-%m-%d %H:%M:%S', '%d/%m/%Y %H:%M:%S'
    ]
    for fmt in formats:
        try:
            return pd.to_datetime(x, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.to_datetime(x, errors='coerce')

if 'date_reported' in df.columns:
    df['date_reported'] = df['date_reported'].apply(parse_date_simple)
    df['date_reported'] = df['date_reported'].fillna(df['date_reported'].mode()[0])

In [72]:
if 'date_reported' in df.columns and df['date_reported'].dtype != 'object':
    df['year_from_date'] = df['date_reported'].dt.year
    inconsistent_years = df[df['year_reported'] != df['year_from_date']]

    # Corrigir: usar ano da data reportada
    if len(inconsistent_years) > 0:
        df['year_reported'] = df['year_from_date']

    df.drop('year_from_date', axis=1, inplace=True)

In [73]:
for col in ['case_id', 'reporter', 'media_outlet', 'file_number', 'investigator_id']:
    df[col] = df[col].astype(str).str.strip().replace({'nan': np.nan, '': np.nan})
    df[col].fillna(f'NA_{col.upper()}')

In [74]:
df['case_description'] = df['case_description'].astype(str).str.strip().replace({'nan': np.nan, '': np.nan})
df['case_description'].fillna('SEM DESCRICAO')

0        Operação policial com buscas e apreensões. Há ...
1        Publicidade de contratos públicos, possível co...
2        Publicidade de contratos públicos, possível co...
3        Caso relativo a fraudes em licitações. Ver det...
4        Acusados negam as acusações. Processo com falt...
                               ...                        
50315    Denúncia sobre adjudicação directa a empresa X...
50316    Publicidade de contratos públicos, possível co...
50317    Acusados negam as acusações. Processo com falt...
50318    Operação policial com buscas e apreensões. Há ...
50319    Denúncia sobre adjudicação directa a empresa X...
Name: case_description, Length: 50300, dtype: object

In [75]:
df['amount_eur'] = pd.to_numeric(df['amount_eur'], errors='coerce')
media_amount = df['amount_eur'].mean()

df['amount_eur'] = df['amount_eur'].fillna(media_amount)
df['amount_eur'] = df['amount_eur'].round(2)

In [87]:
df['investigation_status'] = df['investigation_status'].astype(str).str.lower().str.strip()

df['investigation_status'] = df['investigation_status'].replace({
    'arquivado': 'ARQUIVADO',
    'em investigação': 'EM_INVESTIGACAO',
    'aberto': 'EM_INVESTIGACAO',
    'pending': 'EM_INVESTIGACAO',
    'closed': 'FECHADO',
    'fechado': 'FECHADO',
    'encerrado': 'FECHADO',
    'nan': 'DESCONHECIDO',
    'n/a': 'DESCONHECIDO'
})


In [77]:
for col in ['case_id', 'reporter', 'media_outlet', 'file_number', 'investigator_id']:
    df[col] = df[col].replace('', f'NA_{col.upper()}')            # substitui strings vazias
    df[col] = df[col].fillna(f'NA_{col.upper()}')                 # substitui NaN reais

In [88]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50300 entries, 0 to 50319
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   case_id                   50300 non-null  object        
 1   year_reported             50300 non-null  int32         
 2   region                    50300 non-null  object        
 3   municipality              50300 non-null  object        
 4   sector                    50300 non-null  object        
 5   amount_eur                50300 non-null  float64       
 6   bribe_flag                50300 non-null  object        
 7   bribe_amount              50300 non-null  float64       
 8   investigation_status      50300 non-null  object        
 9   accused_count             50300 non-null  int64         
 10  convicted_flag            50300 non-null  bool          
 11  sentence_years            50300 non-null  float64       
 12  appeal_flag            

In [89]:
print("\n--- Conclusão da Limpeza de Dados ---")
print(f"Linhas após a limpeza e remoção de anos inválidos: {len(df)}")


--- Conclusão da Limpeza de Dados ---
Linhas após a limpeza e remoção de anos inválidos: 50300


> **Inspeção rápida variável a variável (i.e., coluna a coluna)**

In [90]:
df.describe(include='all').round(2)

,case_id,year_reported,region,municipality,sector,amount_eur,bribe_flag,bribe_amount,investigation_status,accused_count,...,sentence_years,appeal_flag,case_description,source_url,reporter,media_outlet,date_reported,file_number,investigator_id,accused_professions_list
count,50300,50300.00,50300,50300,50300,50300.00,50300,50300.00,50300,50300.00,...,50300.00,50300,50300,50300,50300,50300,50300,50300,50300,50300
unique,50000,NaN,10,8,10,NaN,2,NaN,4,NaN,...,NaN,2,21,30323,104,14,NaN,48783,301,800
top,CT-037019,NaN,PORTO,LISBOA,TRANSPORTES,NaN,False,NaN,EM_INVESTIGACAO,NaN,...,NaN,False,Acusados negam as acusações. Processo com falt...,nan,Ana,Diário de Notícias,NaN,NA_FILE_NUMBER,NA_INVESTIGATOR_ID,[JUIZ]
freq,2,NaN,6726,6399,6421,NaN,38730,NaN,23520,NaN,...,NaN,45207,9472,2592,5227,6978,NaN,918,7627,1547
mean,NaN,2009.55,NaN,NaN,NaN,2242156.10,NaN,19958.22,NaN,1.90,...,0.93,NaN,NaN,NaN,NaN,NaN,2010-01-19 07:24:30.632206848,NaN,NaN,NaN
min,NaN,1995.00,NaN,NaN,NaN,-4997108.33,NaN,0.00,NaN,0.00,...,0.00,NaN,NaN,NaN,NaN,NaN,1995-01-01 00:00:00,NaN,NaN,NaN
25%,NaN,2002.00,NaN,NaN,NaN,1437247.11,NaN,0.00,NaN,1.00,...,0.00,NaN,NaN,NaN,NaN,NaN,2002-06-30 00:00:00,NaN,NaN,NaN
50%,NaN,2009.00,NaN,NaN,NaN,2242156.10,NaN,0.00,NaN,2.00,...,0.00,NaN,NaN,NaN,NaN,NaN,2009-08-10 12:00:00,NaN,NaN,NaN
75%,NaN,2017.00,NaN,NaN,NaN,3281684.02,NaN,0.00,NaN,3.00,...,0.00,NaN,NaN,NaN,NaN,NaN,2017-10-14 00:00:00,NaN,NaN,NaN
max,NaN,2025.00,NaN,NaN,NaN,4999879.08,NaN,499857.06,NaN,9.00,...,20.00,NaN,NaN,NaN,NaN,NaN,2025-12-31 00:00:00,NaN,NaN,NaN


In [91]:
(df
 .filter(['year_reported','amount_year','bribe_amount','accused_count','sentence_years'])
 .describe(percentiles=[.01, .25, .5, .75, .99])
 .round(2)
 )

,year_reported,bribe_amount,accused_count,sentence_years
count,50300.00,50300.00,50300.00,50300.00
mean,2009.55,19958.22,1.90,0.93
std,8.89,78193.98,1.07,3.40
min,1995.00,0.00,0.00,0.00
1%,1995.00,0.00,0.00,0.00
25%,2002.00,0.00,1.00,0.00
50%,2009.00,0.00,2.00,0.00
75%,2017.00,0.00,3.00,0.00
99%,2025.00,433988.91,5.00,17.80
max,2025.00,499857.06,9.00,20.00


In [92]:
(df
 .filter(['case_id','region','municipality','sector','investigation_status','case_description','source_url','reporter','media_outler','file_number','investigator_id','accused_professions_list'])
 .astype('object')
 .describe(include=['datetime','object'])
 )

,case_id,region,municipality,sector,investigation_status,case_description,source_url,reporter,file_number,investigator_id,accused_professions_list
count,50300,50300,50300,50300,50300,50300,50300,50300,50300,50300,50300
unique,50000,10,8,10,4,21,30323,104,48783,301,800
top,CT-037019,PORTO,LISBOA,TRANSPORTES,EM_INVESTIGACAO,Acusados negam as acusações. Processo com falt...,nan,Ana,NA_FILE_NUMBER,NA_INVESTIGATOR_ID,[JUIZ]
freq,2,6726,6399,6421,23520,9472,2592,5227,918,7627,1547


In [93]:
df.to_csv('corruption_dataset_LIMPO1.csv', index=False, encoding='utf-8')

In [94]:
print("\n=== VERIFICAÇÃO DE OUTLIERS ===")
numeric_cols = ['amount_eur', 'bribe_amount', 'sentence_years', 'accused_count']
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f"Outliers em {col}: {len(outliers)}")


=== VERIFICAÇÃO DE OUTLIERS ===
Outliers em amount_eur: 1423
Outliers em bribe_amount: 6606
Outliers em sentence_years: 4531
Outliers em accused_count: 37


In [95]:
print("\n=== VERIFICAÇÃO TEMPORAL ===")
if 'date_reported' in df.columns and df['date_reported'].dtype != 'object':
    df['year_from_date'] = df['date_reported'].dt.year
    inconsistent_years = df[df['year_reported'] != df['year_from_date']]
    print(f"Inconsistências ano reportado: {len(inconsistent_years)}")


=== VERIFICAÇÃO TEMPORAL ===
Inconsistências ano reportado: 0


In [97]:
print("\n=== VERIFICAÇÃO FINAL DE VALORES NULOS ===")
null_summary = df.isnull().sum()
null_cols = null_summary[null_summary > 0]
if len(null_cols) > 0:
    print(null_cols)
else:
    print("Nenhum valor nulo encontrado!")


=== VERIFICAÇÃO FINAL DE VALORES NULOS ===
Nenhum valor nulo encontrado!


**TABELA DICIONÁRIO**

| #  |           Nome           | Descrição                                            | Tipo de Dados | Escala  | Formato <br> (em Python) |
|----|:------------------------:|------------------------------------------------------|:-------------:|:-------:|:-----------------------:|
| 0  |         case_id          | Identificador único do caso                          |  Categórico   | Nominal | object
|
| 1  |      year_reported       | Ano em que o caso foi reportado                      |   Numérico    |  Rácio  | int64
|
| 2  |          region          | Região de Portugal onde o caso ocorreu               |  Categórico   | Nominal | object
|
| 3  |       municipality       | Municipio espepcífico do caso                        |  Categórico   | Nominal | object
|
| 4  |          sector          | Setor da economia ou administração pública envolvido |  Categórico   | Nominal | object
|
| 5  |        amount_eur        | Montante total da fraude/irregularidade              |   Numérico    |  Rácio  | float64
|
| 6  |        bribe_flag        | Indicador se houve ou não suborno                    |  Categórico   | Binário | bool
|
| 7  |       bribe_amount       | Montante específico do suborno                       |   Numérico    |  Rácio  | float64
|
| 8  |   investigation_status   | Estado atual da investigação                         |  Categórico   | Binário | bool
|
| 9  |      accused_count       | Número de pessoas acusadas no caso                   |  Categórico   |  Rácio  | int64
|
| 10 | accused_professions_list | Lista de profissões dos acusados                     |  Categórico   | Nominal | object
|
| 11 |      convicted_flag      | Indicador se houve ou não condenação no caso         |  Categórico   | Binário | bool
|
| 12 |      sentence_years      | Anos de sentença atribuidos                          |   Numérico    |  Rácio  | float64
|
| 13 |       appeal_flag        | Indicador se foi ou não apresentado recurso          |  Categórico   | Binário | bool
|
| 14 |     case_description     | Descrição detalhada do caso                          |  Categórico   | Nominal | object
|
| 15 |        source_url        | URL de origem da notícia ou reporte                  |  Categórico   | Nominal | object
|
| 16 |         reporter         | Nome do jornalista ou entidade que reportou o caso   |  Categórico   | Nominal | object
|
| 17 |       media_outlet       | Meio de comunicação que pubicou a noticia            |  Categórico   | Nominal | object
|
| 18 |       file_number        | Número interno do processo/ficheiro                  |  Categórico   | Nominal | object
|
| 19 |     investigator_id      | ID do investigador responsável                       |  Categórico   | Nominal | object
|
|20 | date_reported | Data específica em que o caso foi repprtado | Númerico | Rácio | datetime64[ns]

**SÍNTESE DA QUALIDADE DOS DADOS POR DIMENSÃO**

|       Dimensão       | Comentário                                                                                                                                                                                                                                                                |
|:--------------------:|---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Validade/Integridade | Os dados foram retirados de uma plataforma de IA. Apesar de apresentarem ligações URL, alguns dados podem não ser completamente verdadeiros. Esses dados podem não refletir a realidade de maneira precisa e devem ser analisados como um exercício acadêmico ou experimental. |
|     Conformidade     | Os valores seguem padrões esperados para cada tipo de variável. Contudo, a conformidade pode ser artificial devido à ausência de variações comuns em dados reais.                                                                                                         |
|     Redundância      | Depois de verificação, não foram encontradas linhas duplicadas.                                                                                                                                                                                                           |
|      Completude      | Os dados apresentam valores nulos ou ausentes, refletindo baixa completude. Com isto tentamos completar os dados omissos com informações dadas no dataset.                                                                                                                |
|     Consistência     | As variáveis apresentam consistência formal, com formatos e tipos de dados homogêneos. Por exemplo, as variáveis booleanas estão devidamente codificadas como True ou False.                                                                                              |
|      Atualidade      | Os dados têm relação com períodos específicos, mas alguns já não são muito recentes.                                                                                                                                                                                      |

* Tabela de dados original: **50320** linhas e 20 colunas
* Tabela de dados depois da limpeza inicial: **50300** linhas e 20 colunas